# QueryGenie — Tier-2 reproduction & Week-7 ablations (CodeS on Spider dev)

Team 21 · anchor paper CodeS (arXiv:2402.16347) · **Runtime → Change runtime type → T4 GPU** before running.

Every cell fetches the same self-contained script (`scripts/colab_eval_tier2.py` in the repo, mirrored on a gist). It downloads Spider + the model once per runtime, evaluates all 1034 dev examples with the authors' full database prompt and candidate-selection rule, and prints a `COPY THIS BACK:` line to paste into `results/reproduction_log.csv`.

Env switches: `QG_MODEL` (default `seeklhy/codes-1b-spider`), `QG_RUN_ID`, `QG_ABLATE` (`no_values`, `no_content`, `no_select`, `no_fk`).

## 0 · Check GPU

In [1]:
!nvidia-smi -L
import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__)

GPU 0: Tesla T4 (UUID: GPU-db46c46c-e2ff-fa50-9e0c-a98e4d2916da)
torch 2.11.0+cu128 | transformers 5.16.1


## 1 · Faithful reproduction — CodeS-1B (Table 5 target: 77.9% EX)
T2-002 = full prompt, beam 0 only → 71.7%. T2-003 = + authors' first-executable-candidate rule → 73.8%. Re-run only if needed.

In [2]:
import os, urllib.request; os.environ["QG_RUN_ID"]="T2-003"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

spider_data.zip: reconstructing file:   0%|          |  0.00B /  206MB            

spider_data.zip: downloading bytes:           |  0.00B            

ablations: none
model: seeklhy/codes-1b-spider | run: T2-003
device: cuda | examples: 1034


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.55GB            

model.safetensors: downloading bytes:           |  0.00B            

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

[50/1034] EX=0.8200 exec=1.00 2.0s/ex
SQL execution runtime error: ambiguous column name: petid.
SQL execution runtime error: ambiguous column name: petid.
SQL execution runtime error: ambiguous column name: stuid.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: model.
SQL execution runtime e

## 2 · Week 7 — ablation study (baseline = T2-003, 73.8%)
Remove one component at a time; the drop shows how much it contributes.

### AB-003 · no representative column values in the schema

In [3]:
import os, urllib.request; os.environ["QG_RUN_ID"]="AB-003"; os.environ["QG_ABLATE"]="no_values"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

ablations: ['no_values']
model: seeklhy/codes-1b-spider | run: AB-003
device: cuda | examples: 1034


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

SQL execution runtime error: no such column: singer.average.
SQL execution runtime error: no such column: singer.average.
[50/1034] EX=0.8200 exec=1.00 1.7s/ex
SQL execution runtime error: ambiguous column name: petid.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL exe

### AB-004 · no matched contents (value retriever off)

In [4]:
import os, urllib.request; os.environ["QG_RUN_ID"]="AB-004"; os.environ["QG_ABLATE"]="no_content"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

ablations: ['no_content']
model: seeklhy/codes-1b-spider | run: AB-004
device: cuda | examples: 1034


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[50/1034] EX=0.8200 exec=1.00 1.9s/ex
SQL execution runtime error: ambiguous column name: petid.
SQL execution runtime error: ambiguous column name: petid.
SQL execution runtime error: ambiguous column name: petid.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: weight.
SQL execution runtime error: no such column: model.
[100/1034] EX=0.7300 ex

### AB-005 · no foreign-keys block

In [5]:
import os, urllib.request; os.environ["QG_RUN_ID"]="AB-005"; os.environ["QG_ABLATE"]="no_fk"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

ablations: ['no_fk']
model: seeklhy/codes-1b-spider | run: AB-005
device: cuda | examples: 1034


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[50/1034] EX=0.8800 exec=1.00 1.6s/ex
SQL execution runtime error: no such column: major.
SQL execution runtime error: incomplete input.
SQL execution runtime error: incomplete input.
SQL execution runtime error: no such column: ad.
SQL execution runtime error: incomplete input.
SQL execution runtime error: ambiguous column name: petid.
SQL execution runtime error: ambiguous column name: petid.
SQL execution runtime error: ambiguous column name: stuid.
SQL execution runtime error: ambiguous column name: stuid.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: model.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: horsepower.
SQL execution runtime error: no such column: mod

### AB-006 · beam 0 only (candidate selection off) — re-measures T2-002 with the same code path

In [6]:
import os, urllib.request; os.environ["QG_RUN_ID"]="AB-006"; os.environ["QG_ABLATE"]="no_select"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

ablations: ['no_select']
model: seeklhy/codes-1b-spider | run: AB-006
device: cuda | examples: 1034


Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

[50/1034] EX=0.8200 exec=1.00 1.9s/ex
[100/1034] EX=0.7300 exec=0.94 1.7s/ex
[150/1034] EX=0.6733 exec=0.89 1.7s/ex
[200/1034] EX=0.6900 exec=0.89 1.7s/ex
[250/1034] EX=0.6760 exec=0.88 1.7s/ex
[300/1034] EX=0.7133 exec=0.90 1.6s/ex
[350/1034] EX=0.7229 exec=0.90 1.6s/ex
[400/1034] EX=0.7400 exec=0.91 1.6s/ex
[450/1034] EX=0.7511 exec=0.92 1.6s/ex
[500/1034] EX=0.7240 exec=0.92 1.8s/ex
[550/1034] EX=0.7273 exec=0.93 2.1s/ex
[600/1034] EX=0.7283 exec=0.93 2.2s/ex
[650/1034] EX=0.7323 exec=0.93 2.2s/ex
[700/1034] EX=0.7414 exec=0.93 2.1s/ex
[750/1034] EX=0.7307 exec=0.93 2.1s/ex
[800/1034] EX=0.6987 exec=0.92 2.1s/ex
[850/1034] EX=0.7024 exec=0.92 2.0s/ex
[900/1034] EX=0.7089 exec=0.93 2.0s/ex
[950/1034] EX=0.7084 exec=0.93 2.1s/ex
[1000/1034] EX=0.7090 exec=0.93 2.2s/ex
[1034/1034] EX=0.7166 exec=0.94 2.2s/ex

==== FULL CodeS prompt | Spider dev EX = 0.7166 (741/1034) | executes 0.935 | 2265s ====
COPY THIS BACK:  AB-006 seeklhy/codes-1b-spider ablate=no_select EX=0.7166 (741/1034) exec

## 3 · Larger model — CodeS-3B (Table 5 target: 83.4% EX)
Same pipeline, bigger checkpoint (~6 GB fp16, fits a T4). ~1.5–2 h.

In [ ]:
import os, urllib.request; os.environ["QG_RUN_ID"]="T2-004"; os.environ["QG_MODEL"]="seeklhy/codes-3b-spider"; exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())

ablations: ['no_select']
model: seeklhy/codes-3b-spider | run: T2-004
device: cuda | examples: 1034


config.json:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/717 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.06M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json:   0%|          | 0.00/34.4k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/437 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

[50/1034] EX=0.9400 exec=1.00 4.0s/ex
[100/1034] EX=0.8700 exec=0.97 3.7s/ex
[150/1034] EX=0.8067 exec=0.95 3.9s/ex
[200/1034] EX=0.7800 exec=0.96 3.9s/ex
[250/1034] EX=0.7760 exec=0.96 3.7s/ex
[300/1034] EX=0.7900 exec=0.96 3.5s/ex
[350/1034] EX=0.7971 exec=0.96 3.5s/ex
[400/1034] EX=0.8050 exec=0.96 3.4s/ex
[450/1034] EX=0.8133 exec=0.96 3.4s/ex
[500/1034] EX=0.7780 exec=0.96 3.7s/ex
[550/1034] EX=0.7655 exec=0.96 4.3s/ex
[600/1034] EX=0.7667 exec=0.96 4.7s/ex
[650/1034] EX=0.7692 exec=0.97 4.6s/ex
[700/1034] EX=0.7814 exec=0.97 4.4s/ex
[750/1034] EX=0.7800 exec=0.97 4.4s/ex
[800/1034] EX=0.7512 exec=0.97 4.4s/ex
[850/1034] EX=0.7529 exec=0.97 4.3s/ex
[900/1034] EX=0.7567 exec=0.97 4.2s/ex


## 4 · Harness v2 (after the 26 Sep audit) — see `results/harness_audit.md`
Fixes: numeric values in the value retriever, tables.json column comments, authors' truncation, saved predictions, and **scoring with the official Spider evaluator** (the metric the paper uses). The `OFFICIAL_EX` number is the one to compare with 77.9%.

In [ ]:
import os, urllib.request; os.environ["QG_RUN_ID"]="T2-005"; os.environ.pop("QG_ABLATE",None); os.environ.pop("QG_MODEL",None); exec(urllib.request.urlopen("https://gist.githubusercontent.com/jayakanthh/22c5b86fa8b48430dae48f86d3da2520/raw/colab_eval_tier2.py").read().decode())